# Download, filter, and visualize extracted residual-stream activations

This notebook works with the chunk files produced by `scripts/extract_residual_stream_positions_from_gcs.py`. It downloads the completions index and extracted chunks, selects samples using prompt metadata, loads one residual-stream layer and cached token position, projects the activations with PCA, and visualizes the result.

## 1. Setup

For a fresh Colab runtime, uncomment the clone and authentication commands. Local runs can skip them.

In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
%cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

In [ ]:
import gc
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from google.cloud import storage
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

## 2. Configure the workflow

Metadata filters use the same semantics as `download_filtered_activations.ipynb`: `None` disables a template filter, and every entry in `TASK_METADATA_FILTERS` must match. Set `LAYER_NAME` to `None` to use the final available residual-stream layer. `POSITION_INDEX` refers to the cached indices written by the extractor (0, 1, or 2), not an absolute token index.

In [ ]:
PROMPT_FRAMING: str | None = 'task_available_time'
OUTPUT_FORMAT: str | None = None
TASK_METADATA_FILTERS: dict[str, object] | None = {'domain': 'communication'}

ACTIVATIONS_GCS_URI = 'gs://temporal-research-bucket/resid_only_0_2'
COMPLETIONS_GCS_URI = 'gs://temporal-research-bucket/completions/completions_256.jsonl'
DATA_DIR = repo_root / 'data' / 'filtered_residual_stream_activations'
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
OVERWRITE = False
MAX_SAMPLES: int | None = None
# None reads every downloaded chunk. Otherwise list local filenames, for example:
# SELECTED_CHUNK_FILES = ['residual_stream_positions_0_2_chunk_00000.pt']
SELECTED_CHUNK_FILES: list[str] | None = None

LAYER_NAME: str | None = None
POSITION_INDEX = 0
PCA_MODE = 'matched'  # matched, residualized, or raw
PHRASING_FIELDS = ['template_metadata.prompt_framing', 'template_metadata.output_format']
SEMANTIC_FIELDS = ['task', 'base_value', 'base_unit']

## 3. Download the extracted chunks and completions index

All extracted `.pt` chunks are cached locally and skipped on subsequent runs unless `OVERWRITE` is true. File selection happens only after the complete download, so changing `SELECTED_CHUNK_FILES` never requires another GCS listing or download.

In [ ]:
def parse_gcs_uri(uri):
    if not uri.startswith('gs://'):
        raise ValueError(f'Expected a gs:// URI, got {uri!r}')
    bucket, separator, object_name = uri[5:].partition('/')
    if not bucket or not separator or not object_name.strip('/'):
        raise ValueError(f'GCS URI must contain a bucket and object/prefix: {uri!r}')
    return bucket, object_name.strip('/')


client = storage.Client(project=PROJECT_ID)
DATA_DIR.mkdir(parents=True, exist_ok=True)

completions_bucket, completions_object = parse_gcs_uri(COMPLETIONS_GCS_URI)
COMPLETIONS_PATH = DATA_DIR / Path(completions_object).name
if OVERWRITE or not COMPLETIONS_PATH.exists():
    client.bucket(completions_bucket).blob(completions_object).download_to_filename(str(COMPLETIONS_PATH))

activation_bucket, activation_prefix = parse_gcs_uri(ACTIVATIONS_GCS_URI)
blobs = sorted(
    (blob for blob in client.bucket(activation_bucket).list_blobs(prefix=activation_prefix.rstrip('/') + '/') if blob.name.endswith('.pt')),
    key=lambda blob: blob.name,
)
if not blobs:
    raise FileNotFoundError(f'No .pt chunks found below {ACTIVATIONS_GCS_URI}')

chunk_dir = DATA_DIR / 'chunks'
chunk_dir.mkdir(exist_ok=True)
downloaded_chunk_paths = []
for blob in tqdm(blobs, desc='Downloading chunks'):
    destination = chunk_dir / Path(blob.name).name
    if OVERWRITE or not destination.exists():
        blob.download_to_filename(str(destination))
    downloaded_chunk_paths.append(destination)

print(f'Completions: {COMPLETIONS_PATH}')
print(f'Downloaded/local chunks: {len(downloaded_chunk_paths):,}')

### 3.1 Select local chunk files to read

Set `SELECTED_CHUNK_FILES` in the configuration cell to a list of basenames, or leave it as `None` to read every downloaded chunk. This selection does not delete or move any downloaded file.

In [ ]:
downloaded_by_name = {path.name: path for path in downloaded_chunk_paths}
if SELECTED_CHUNK_FILES is None:
    selected_chunk_paths = downloaded_chunk_paths
else:
    missing_files = sorted(set(SELECTED_CHUNK_FILES) - set(downloaded_by_name))
    if missing_files:
        raise FileNotFoundError(f'Selected chunk files were not downloaded: {missing_files}')
    selected_chunk_paths = [downloaded_by_name[name] for name in SELECTED_CHUNK_FILES]

if not selected_chunk_paths:
    raise ValueError('No local chunk files were selected.')
print(f'Selected {len(selected_chunk_paths):,} of {len(downloaded_chunk_paths):,} local chunks to read.')
for path in selected_chunk_paths[:10]:
    print(path)

## 4. Filter completion records

The JSONL line number is the sample index stored in each extracted chunk. Retain both the selected index set and prompt metadata so activation rows can be joined without relying on chunk order.

In [ ]:
selected_metadata = {}
with COMPLETIONS_PATH.open(encoding='utf-8') as completion_file:
    for sample_index, line in enumerate(completion_file):
        record = json.loads(line)
        metadata = record['prompt_metadata']
        template = metadata.get('template_metadata', {})
        task_metadata = metadata.get('task_metadata', {})
        template_matches = (
            (PROMPT_FRAMING is None or template.get('prompt_framing') == PROMPT_FRAMING)
            and (OUTPUT_FORMAT is None or template.get('output_format') == OUTPUT_FORMAT)
        )
        task_matches = all(
            task_metadata.get(key) == value
            for key, value in (TASK_METADATA_FILTERS or {}).items()
        )
        if template_matches and task_matches:
            selected_metadata[sample_index] = metadata
            if MAX_SAMPLES is not None and len(selected_metadata) >= MAX_SAMPLES:
                break

if not selected_metadata:
    raise ValueError('No completion records matched the configured filters.')
selected_indices = set(selected_metadata)
print(f'Selected {len(selected_indices):,} completion records.')
print('First sample indices:', sorted(selected_indices)[:10])

## 5. Load one layer and cached position

Only matching rows from the chosen layer are copied out of each chunk. Other residual layers are released before the next chunk is loaded, keeping retained memory proportional to the filtered feature matrix.

In [ ]:
def layer_sort_key(name):
    prefix, separator, suffix = name.rpartition('/')
    return (prefix, int(suffix)) if separator and suffix.isdigit() else (name, name)


preview = torch.load(selected_chunk_paths[0], map_location='cpu', weights_only=True)
residuals = preview.get('residual_stream_activations')
if not isinstance(residuals, dict) or not residuals:
    raise ValueError(f'{selected_chunk_paths[0]} has no residual_stream_activations mapping.')
available_layers = sorted(residuals, key=layer_sort_key)
selected_layer = LAYER_NAME or available_layers[-1]
if selected_layer not in residuals:
    raise ValueError(f'Layer {selected_layer!r} is unavailable. Choose from {available_layers}.')
del residuals, preview
gc.collect()
print('Available layers:', available_layers)
print(f'Loading layer={selected_layer!r}, cached position={POSITION_INDEX}.')

feature_parts = []
loaded_indices = []
absolute_positions = []
for path in tqdm(selected_chunk_paths, desc='Filtering activation rows'):
    payload = torch.load(path, map_location='cpu', weights_only=True)
    chunk_indices = [int(index) for index in payload['sample_indices']]
    row_offsets = [offset for offset, index in enumerate(chunk_indices) if index in selected_indices]
    if row_offsets:
        cached_indices = list(payload['cached_position_indices'])
        if POSITION_INDEX not in cached_indices:
            raise ValueError(f'{path} does not contain cached position {POSITION_INDEX}.')
        position_offset = cached_indices.index(POSITION_INDEX)
        tensor = payload['residual_stream_activations'][selected_layer]
        rows = torch.as_tensor(row_offsets, dtype=torch.long)
        feature_parts.append(tensor.index_select(0, rows)[:, position_offset, :].to(torch.float32).contiguous())
        loaded_indices.extend(chunk_indices[offset] for offset in row_offsets)
        chunk_absolute_positions = payload['absolute_token_positions']
        absolute_positions.extend(chunk_absolute_positions[offset][position_offset] for offset in row_offsets)
    del payload
    gc.collect()

if not feature_parts:
    raise ValueError('None of the selected completion indices occur in the downloaded chunks.')
activation_matrix = torch.cat(feature_parts, dim=0)
del feature_parts
print('Activation matrix shape:', tuple(activation_matrix.shape))
print(f'Loaded {len(loaded_indices):,} of {len(selected_indices):,} selected samples.')

## 6. Prepare metadata and project with PCA

`matched` averages records that share the configured semantic fields before PCA, reducing prompt-phrasing variation. `residualized` removes additive categorical phrasing effects while retaining every row. `raw` performs ordinary centered PCA.

In [ ]:
def flatten_scalar_metadata(metadata, prefix=''):
    flattened = {}
    for key, value in metadata.items():
        path = f'{prefix}.{key}' if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_scalar_metadata(value, path))
        elif not isinstance(value, (list, tuple, set)):
            flattened[path] = value
    return flattened


metadata_df = pd.DataFrame([flatten_scalar_metadata(selected_metadata[index]) for index in loaded_indices])
metadata_df.insert(0, 'sample_index', loaded_indices)
metadata_df.insert(1, 'absolute_token_position', absolute_positions)
available_phrasing_fields = [field for field in PHRASING_FIELDS if field in metadata_df]
available_semantic_fields = [field for field in SEMANTIC_FIELDS if field in metadata_df]

if PCA_MODE == 'matched':
    if not available_semantic_fields:
        raise ValueError('Matched PCA requires at least one available SEMANTIC_FIELDS entry.')
    vectors, rows = [], []
    groups = metadata_df.groupby(available_semantic_fields, dropna=False, sort=False).indices
    for row_offsets in groups.values():
        row_offsets = list(row_offsets)
        vectors.append(activation_matrix.index_select(0, torch.as_tensor(row_offsets)).mean(dim=0))
        row = metadata_df.iloc[row_offsets[0]].copy()
        row['source_sample_count'] = len(row_offsets)
        for field in available_phrasing_fields:
            if metadata_df.iloc[row_offsets][field].nunique(dropna=True) > 1:
                row[field] = '<averaged>'
        rows.append(row)
    pca_matrix = torch.stack(vectors)
    analysis_metadata_df = pd.DataFrame(rows).reset_index(drop=True)
elif PCA_MODE == 'residualized':
    design = pd.get_dummies(metadata_df[available_phrasing_fields].fillna('<missing>'), dtype=float, drop_first=True)
    design.insert(0, 'intercept', 1.0)
    design_tensor = torch.as_tensor(design.to_numpy(), dtype=activation_matrix.dtype)
    pca_matrix = activation_matrix - design_tensor @ torch.linalg.lstsq(design_tensor, activation_matrix).solution
    analysis_metadata_df = metadata_df.copy()
    analysis_metadata_df['source_sample_count'] = 1
elif PCA_MODE == 'raw':
    pca_matrix = activation_matrix
    analysis_metadata_df = metadata_df.copy()
    analysis_metadata_df['source_sample_count'] = 1
else:
    raise ValueError("PCA_MODE must be 'matched', 'residualized', or 'raw'.")

if len(pca_matrix) < 3:
    raise ValueError('At least three analysis rows are required for a 3D PCA projection.')
centered = pca_matrix - pca_matrix.mean(dim=0, keepdim=True)
u, s, _v = torch.pca_lowrank(centered, q=3, center=False)
projections = u * s
explained_fraction = (s.square() / centered.square().sum()).cpu().numpy()
print('PCA input shape:', tuple(pca_matrix.shape))
print('Explained variance fractions:', explained_fraction)

## 7. Visualize the projection

The controls discover scalar prompt-metadata fields automatically. Select a color field and optionally restrict the plot to one or more values of another field.

In [ ]:
unit_to_months = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1, 'year': 12, 'decade': 120, 'century': 1200, 'millennium': 12000,
}
unit_to_months.update({f'{unit}s': value for unit, value in list(unit_to_months.items())})
unit_to_months['centuries'] = 1200
unit_to_months['millennia'] = 12000

df_projs = analysis_metadata_df.copy().reset_index(drop=True)
df_projs[['PC1', 'PC2', 'PC3']] = projections.cpu().numpy()
value_field = 'base_value' if 'base_value' in df_projs else 'value'
unit_field = 'base_unit' if 'base_unit' in df_projs else 'unit'
df_projs['time_horizon_months'] = [float(value) * unit_to_months[str(unit).lower()] for value, unit in zip(df_projs[value_field], df_projs[unit_field])]
df_projs['log10_time_horizon_months'] = np.log10(df_projs['time_horizon_months'])
metadata_fields = sorted(column for column in df_projs if column not in {'PC1', 'PC2', 'PC3', 'sample_index'})
color_fields = ['log10_time_horizon_months', *[field for field in metadata_fields if field != 'log10_time_horizon_months']]
print(f'Prepared {len(df_projs):,} projected points.')

In [ ]:
import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
from pandas.api.types import is_bool_dtype, is_numeric_dtype

color_dropdown = widgets.Dropdown(options=color_fields, value='log10_time_horizon_months', description='Color by:', layout=widgets.Layout(width='500px'))
filter_dropdown = widgets.Dropdown(options=[('(no filter)', None), *[(field, field) for field in metadata_fields]], description='Filter by:', layout=widgets.Layout(width='500px'))
filter_values = widgets.SelectMultiple(description='Keep:', rows=6, layout=widgets.Layout(width='500px'))
plot_output = widgets.Output()

def update_filter_values():
    field = filter_dropdown.value
    values = [] if field is None else sorted(df_projs[field].dropna().unique().tolist(), key=str)
    filter_values.options = [(str(value), value) for value in values]
    filter_values.value = tuple(values)
    filter_values.disabled = field is None

def render_plot(change=None):
    filtered = df_projs
    if filter_dropdown.value is not None:
        filtered = filtered[filtered[filter_dropdown.value].isin(filter_values.value)]
    plot_output.clear_output(wait=True)
    with plot_output:
        if filtered.empty:
            print('No points match the selected filter values.')
            return
        color_field = color_dropdown.value
        plot_data = filtered.copy()
        numeric_color = is_numeric_dtype(plot_data[color_field]) and not is_bool_dtype(plot_data[color_field])
        if not numeric_color:
            plot_data[color_field] = plot_data[color_field].astype('string').fillna('<missing>')
        fig = px.scatter_3d(plot_data, x='PC1', y='PC2', z='PC3', color=color_field, color_continuous_scale='Viridis' if numeric_color else None, hover_data=['sample_index', 'time_horizon_months'], title=f'{selected_layer}, cached position {POSITION_INDEX} ({len(plot_data):,} points)', opacity=0.7)
        fig.update_traces(marker={'size': 4})
        display(fig)

def on_filter_change(change):
    update_filter_values()
    render_plot()

color_dropdown.observe(render_plot, names='value')
filter_dropdown.observe(on_filter_change, names='value')
filter_values.observe(render_plot, names='value')
update_filter_values()
display(widgets.VBox([color_dropdown, filter_dropdown, filter_values, plot_output]))
render_plot()

In [ ]:
df_projs